# Kaggle 재현 실험: 소형 언어 모델 3종

이 노트북은 논문/보고서 독자가 Kaggle GPU 환경에서 본 연구의 소형 모델 실험을 직접 재현할 수 있도록 만든 실행 가이드입니다.

실험 내용:
- KLUE-BERT, KLUE-RoBERTa, KCBERT를 fine-tuning합니다.
- 원본 test 성능과 변형 test 성능을 비교합니다.
- F1, Delta F1, accuracy, confusion matrix 관련 값을 CSV로 저장합니다.
- 결과 그래프를 `results/figures/`에 저장합니다.

Kaggle 설정:
- Accelerator: GPU 또는 GPU T4 x2
- Internet: On
- 처음 실행은 `--all_variants` 없이 진행하는 것을 권장합니다.

최종 산출물:
- `results/metrics/klue-bert_results.csv`
- `results/metrics/klue-roberta_results.csv`
- `results/metrics/kcbert_results.csv`
- `results/figures/*.png`
- `results_small_models.zip`

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = 'https://github.com/JH-dev1125/korean-adversarial-nlp.git'
PROJECT_DIR = Path('/kaggle/working/korean-adversarial-nlp')

# 새 Kaggle Notebook에서 실행하는 경우 repo를 자동으로 clone합니다.
# 이미 프로젝트 폴더가 있으면 clone을 건너뜁니다.
if not PROJECT_DIR.exists():
    os.chdir('/kaggle/working')
    subprocess.run(['git', 'clone', REPO_URL], check=True)

os.chdir(PROJECT_DIR)
print('프로젝트 경로:', PROJECT_DIR)
print('현재 파일 목록:')
!ls -la

In [ ]:
# GPU가 켜져 있는지 확인합니다.
!nvidia-smi

In [ ]:
# Kaggle 기본 환경에는 최신 peft가 미리 설치되어 있을 수 있습니다.
# transformers==4.40.0과 최신 peft가 충돌하면 Trainer import가 실패하므로 peft를 제거합니다.
# dependency conflict 경고가 보일 수 있지만, 아래 버전 확인 셀이 통과하면 계속 진행해도 됩니다.
!pip uninstall -y -q peft
!pip install -q transformers==4.40.0 datasets==2.19.0 pandas scikit-learn matplotlib seaborn 'tqdm>=4.66.3'

In [ ]:
# 패키지와 GPU 상태를 확인합니다.
# import 오류가 없고 cuda가 True이면 다음 단계로 진행합니다.
import torch
import transformers
import datasets
import pandas as pd
import numpy as np
import sklearn

print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('datasets:', datasets.__version__)
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('scikit-learn:', sklearn.__version__)
print('cuda 사용 가능:', torch.cuda.is_available())

In [ ]:
# 학습에 필요한 processed 데이터와 평가에 필요한 augmented 데이터를 확인합니다.
from pathlib import Path

required_files = [
    Path('data/processed/train.csv'),
    Path('data/processed/val.csv'),
    Path('data/processed/test.csv'),
]

for file_path in required_files:
    print(file_path, '존재' if file_path.exists() else '없음')

augmented_files = sorted(Path('data/augmented').glob('test_*.csv'))
print('변형 test 파일 수:', len(augmented_files))
print('예시:', [p.name for p in augmented_files[:5]])

In [ ]:
# processed 데이터가 없다면 raw 데이터에서 전처리를 수행합니다.
# 이미 data/processed/*.csv를 업로드했다면 이 셀은 자동으로 건너뜁니다.
from pathlib import Path

processed_ready = all(Path(p).exists() for p in [
    'data/processed/train.csv',
    'data/processed/val.csv',
    'data/processed/test.csv',
])

if processed_ready:
    print('processed 데이터가 이미 있어 전처리를 건너뜁니다.')
else:
    print('processed 데이터가 없어 전처리를 실행합니다.')
    !python src/utils/preprocess.py

In [ ]:
# 변형 test 데이터가 없다면 단일 공격 9종 x 강도 3단계 데이터를 생성합니다.
# 이미 data/augmented/test_*.csv를 업로드했다면 이 셀은 자동으로 건너뜁니다.
from pathlib import Path

augmented_files = sorted(Path('data/augmented').glob('test_*.csv'))
if augmented_files:
    print(f'변형 test 파일 {len(augmented_files)}개가 이미 있어 공격 생성을 건너뜁니다.')
else:
    print('변형 test 파일이 없어 공격 데이터를 생성합니다.')
    !python src/attacks/run_all_attacks.py

In [ ]:
# 소형 모델 3개를 순서대로 학습하고 원본 test/변형 test 평가 결과를 저장합니다.
# 기본값은 variant_id=1만 평가합니다.
# 모든 variant를 평가하려면 뒤에 --all_variants를 붙이세요.
# 결과: results/metrics/klue-bert_results.csv 등
!python src/models/small_model.py --model all

In [ ]:
# 저장된 metrics CSV를 확인합니다.
import pandas as pd
from pathlib import Path

metric_files = sorted(Path('results/metrics').glob('*_results.csv'))
print('결과 파일:', [p.name for p in metric_files])

if metric_files:
    preview = pd.concat([pd.read_csv(p).head(3) for p in metric_files], ignore_index=True)
    display(preview)

In [ ]:
# 결과 그래프를 생성합니다.
# 결과: results/figures/*.png
!python src/evaluation/visualize.py

In [ ]:
# 생성된 그래프 파일을 확인합니다.
from pathlib import Path

figure_files = sorted(Path('results/figures').glob('*.png'))
print('그래프 파일:', [p.name for p in figure_files])

In [ ]:
# Kaggle Output에서 내려받기 쉽도록 결과 폴더를 압축합니다.
!zip -r results_small_models.zip results/metrics results/figures